In [ ]:
!pip install bioservices
!pip install biomodels

In [ ]:
import biomodels
from bioservices import BioModels
from pathlib import Path
import shutil

In [ ]:
#récupération des IDs et stockage dans un dictionnaire

disease = ["covid", "dengue", "chikungunya", "Mpox", "west nile", "influenza", "tuberculosis"]
s = BioModels()
model_ids = {}

for el in disease:
    results = s.search(el)
    model_ids[el] = [res['id'] for res in results.get('models', [])]

print(model_ids)


In [ ]:
metadata = biomodels.get_metadata("BIOMD0000000970")
metadata

print(metadata[11])

#biomodels.get_file("metadata.rdf", model_id="BIOMD0000000528")

In [ ]:
#récupération des fichiers SBML/XML/OMEX (Open Modeling EXchange format)
for d in disease :
    if d in model_ids:
      for el in model_ids[d]:
          try:
              metadata = biomodels.get_metadata(el)
              print(f"Modèle {el} pour {d} récupéré.")
              biomodels.get_file(metadata[0])
          except Exception as e:
                print(f"Erreur sur le modèle {el}: {e}")

In [ ]:
racine_data = Path("/../BioModels_Data")
racine_data.mkdir(parents=True, exist_ok=True)

s = BioModels()
diseases = ["covid", "dengue", "chikungunya", "Mpox", "west nile", "influenza"]

for disease in diseases:
    dossier_maladie = racine_data / disease
    dossier_maladie.mkdir(parents=True, exist_ok=True)

    print(f"\n--- Recherche pour : {disease} ---")
    results = s.search(disease)
    ids = [res['id'] for res in results.get('models', [])]

    for model_id in ids:
        try:
            files_objects = biomodels.get_metadata(model_id)
            if not files_objects:
                continue
            target_file_obj = files_objects[0]
            nom_reel = getattr(target_file_obj, 'name', str(target_file_obj)
            result = biomodels.get_file(target_file_obj)
            dest_path = dossier_maladie / nom_reel

            if isinstance(result, (str, Path)) and Path(result).exists():
                shutil.copy(result, dest_path)
                print(f"OK {nom_reel} copié dans /{disease}")
            else:
                with open(dest_path, "w", encoding="utf-8") as f:
                    f.write(str(result))
                print(f"OK {nom_reel} écrit dans /{disease}")

        except Exception as e:
            print(f"Erreur sur {model_id}: {e}")

In [ ]:
import biomodels
from bioservices import BioModels
from pathlib import Path
import shutil

# 1. Dossier racine
racine_data = Path("/../BioModels_Data_DOID")
racine_data.mkdir(parents=True, exist_ok=True)

s = BioModels()

# Dictionnaire de vos DOID (Clé = Nom du dossier, Valeur = l'identifiant DOID)
# Exemple : DOID:0080600 (COVID-19), DOID:12205 (Dengue), etc.
doid_list = {
    "covid": "DOID:0080600",
    "dengue": "DOID:0050125",
    "chikungunya": "DOID:0050012",
    "Mpox": "DOID:3292",
    "west_nile": "DOID:2366",
    "influenza": "DOID:8469"
}

for name, doid in doid_list.items():
    dossier_maladie = racine_data / name
    dossier_maladie.mkdir(parents=True, exist_ok=True)

    print(f"\n--- Recherche pour {name} ({doid}) ---")

    # Recherche spécifique par DOID
    results = s.search(f"id:{doid}")
    ids = [res['id'] for res in results.get('models', [])]

    if not ids:
        print(f"Aucun modèle trouvé pour le DOID {doid}")
        continue

    for model_id in ids:
        try:
            files_objects = biomodels.get_metadata(model_id)
            if not files_objects:
                continue

            target_file_obj = files_objects[0]
            # Extraction sécurisée du nom de fichier
            nom_reel = getattr(target_file_obj, 'name', str(target_file_obj))

            result = biomodels.get_file(target_file_obj)
            dest_path = dossier_maladie / nom_reel

            if isinstance(result, (str, Path)) and Path(result).exists():
                shutil.copy(result, dest_path)
            else:
                with open(dest_path, "w", encoding="utf-8") as f:
                    f.write(str(result))

            print(f"{nom_reel} enregistré.")

        except Exception as e:
            print(f"Erreur sur {model_id}: {e}")